In [191]:
%pip install pandas openpyxl pyarrow matplotlib seaborn scikit-learn statsmodels

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [243]:
import pandas as pd

df = pd.read_excel("../data/Modified_loan_dataset_investor.xlsx")
df.to_parquet("../data/loans.parquet")   # use this from now on

In [244]:
df.shape


(843755, 22)

In [245]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 843755 entries, 0 to 843754
Data columns (total 22 columns):
 #   Column                        Non-Null Count   Dtype         
---  ------                        --------------   -----         
 0   loan_id                       843755 non-null  str           
 1   country                       843755 non-null  str           
 2   loan_issued_at                843755 non-null  datetime64[us]
 3   early_repaid_at               252308 non-null  object        
 4   issued_amount                 843755 non-null  float64       
 5   loan_status                   843755 non-null  str           
 6   initial_interest_rate         843755 non-null  float64       
 7   nr_of_payments                843755 non-null  int64         
 8   principal_balance             836042 non-null  float64       
 9   principal_debt                836042 non-null  float64       
 10  principal_paid_total          836042 non-null  float64       
 11  interest_paid_total     

In [246]:
df["loan_issued_at"].dt.year.value_counts().sort_index()       # To See how many loans each year has

loan_issued_at
2009       667
2010      1163
2011       451
2012       457
2013      2519
2014      8713
2015     14338
2016     10539
2017     17990
2018     25359
2019     56506
2020     27874
2021     51742
2022     63594
2023     81556
2024    147255
2025    182056
2026    150976
Name: count, dtype: int64

In [247]:
df = df[(df["loan_issued_at"] >= "2009-01-01") & (df["loan_issued_at"] < "2026-01-01")].copy()   #Keep 2009–2025 only 

**we removed all loans issued in 2026. The 2026 loans are new and most have not been running long enough to show whether they will default. so including them would make the default rate look lower than it really is**

In [248]:
df.shape

(692779, 22)

In [249]:
print("First loan:", df["loan_issued_at"].min())     # Checking Min loan issued date
print("Last loan:", df["loan_issued_at"].max())      # Checkinhg Max loan issued date

First loan: 2009-02-28 18:05:13
Last loan: 2025-12-31 23:26:15.567000


In [250]:
print(df["is_default"].unique())

[ 0.  1. nan]


In [251]:
print(df["has_default_within_12_months"].unique())

[ 0.  1. nan]


In [252]:
print(df["customer_risk_rating"].unique())

<ArrowStringArray>
[nan, 'D', 'AA', 'F', 'A', 'C', 'B', 'E', 'G', 'HR']
Length: 10, dtype: str


In [253]:
df["is_default"] = df["is_default"].astype("Int8")
df["has_default_within_12_months"] = df["has_default_within_12_months"].astype("Int8")

for col in ["country", "loan_status", "loan_status_risk", "customer_risk_rating"]:
    df[col] = df[col].astype("category")

df.info()

<class 'pandas.DataFrame'>
Index: 692779 entries, 0 to 843754
Data columns (total 22 columns):
 #   Column                        Non-Null Count   Dtype         
---  ------                        --------------   -----         
 0   loan_id                       692779 non-null  str           
 1   country                       692779 non-null  category      
 2   loan_issued_at                692779 non-null  datetime64[us]
 3   early_repaid_at               229082 non-null  object        
 4   issued_amount                 692779 non-null  float64       
 5   loan_status                   692779 non-null  category      
 6   initial_interest_rate         692779 non-null  float64       
 7   nr_of_payments                692779 non-null  int64         
 8   principal_balance             685066 non-null  float64       
 9   principal_debt                685066 non-null  float64       
 10  principal_paid_total          685066 non-null  float64       
 11  interest_paid_total          

In [254]:
order = ["AA", "A", "B", "C", "D", "E", "F", "G", "HR"]  # replace with your actual values
df["customer_risk_rating"] = pd.Categorical(df["customer_risk_rating"], categories=order, ordered=True)

**customer_risk_rating grades each customer at the time the loan was issued. **AA is the best grade** (the most reliable customers, least likely to default) and **HR is the lowest grade** (the riskiest customers, most likely to default)**.

**We converted the column to an **ordered category**, from best to lowest grade:**

**AA → A → B → C → D → E → F → G → HR**

In [255]:
df = df.drop(columns=["nr_of_payments"])

In [256]:
df.shape

(692779, 21)

**nr_of_payments was removed because it is only filled on the day a payment is due, so its value depends on the data extraction date rather than on the loan itself. It carries no information about default risk, and months_on_book already captures how far along each loan is**


In [257]:
df["is_default"].isna().sum()

np.int64(7713)

In [258]:
df["combined_income"].isna().sum()

np.int64(273154)

In [259]:
df["customer_risk_rating"].isna().sum()


np.int64(338379)

In [260]:
df["has_default_within_12_months"].isna().sum()

np.int64(37141)

In [261]:
df["months_in_default"].isna().sum()

np.int64(578825)

**The total dataset is (692779), but when we check here at months_in_defalt, it shows that missing values are (578825). But none is not default. Blank means “not in default” rather than “unknown”. As per the dataset dictionary, the number of months since default occurred, NULL if not default. Amount of months in default.**

In [262]:
df.describe()

,loan_issued_at,issued_amount,initial_interest_rate,principal_balance,principal_debt,principal_paid_total,interest_paid_total,is_default,months_in_default,months_on_book,repaid_amount_total,initial_loan_duration,combined_income,has_default_within_12_months
count,692779,692779.000000,692779.000000,685066.000000,685066.000000,685066.000000,685066.000000,685066.0,113954.000000,684291.000000,685066.000000,692779.000000,419625.000000,655638.0
mean,2022-10-08 22:38:25.429130,2240.389133,0.219801,832.239037,316.804293,1244.026505,559.971501,0.16634,33.997087,18.782329,1901.589952,54.819020,2543.360507,0.13827
min,2009-02-28 18:05:13,6.390000,0.010000,-34.200000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,1.000000,0.000000,0.0
25%,2021-03-24 12:58:56.500000,530.000000,0.174800,0.000000,0.000000,166.190000,50.180000,0.0,9.000000,7.000000,360.880000,36.000000,1679.460000,0.0
50%,2023-10-27 07:32:03.897000,1500.000000,0.187700,0.000000,0.000000,531.000000,207.060000,0.0,20.000000,14.000000,980.595000,60.000000,2327.570000,0.0
75%,2025-01-21 12:02:22.927000,3632.000000,0.229900,915.160000,0.000000,1650.000000,673.200000,0.0,50.000000,25.000000,2660.727500,60.000000,3032.030000,0.0
max,2025-12-31 23:26:15.567000,20000.000000,2.643100,19744.280000,19744.280000,20000.000000,34494.530000,1.0,202.000000,161.000000,41304.710000,120.000000,177871.700000,1.0
std,NaN,2256.226171,0.140999,1653.253813,1045.702744,1687.552648,975.148746,0.372386,32.854720,16.886778,2382.181949,25.289017,1604.809517,0.345183


In [263]:
df.isna().mean().sort_values(ascending=False)   # missing values in Percentage in Descending order

months_in_default               0.835512
next_payment_date_local         0.679416
early_repaid_at                 0.669329
customer_risk_rating            0.488437
combined_income                 0.394287
has_default_within_12_months    0.053612
months_on_book                  0.012252
is_default                      0.011133
interest_paid_total             0.011133
principal_balance               0.011133
repaid_amount_total             0.011133
principal_paid_total            0.011133
principal_debt                  0.011133
loan_status_risk                0.000003
loan_issued_at                  0.000000
country                         0.000000
loan_id                         0.000000
issued_amount                   0.000000
loan_status                     0.000000
initial_interest_rate           0.000000
initial_loan_duration           0.000000
dtype: float64

In [264]:
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(1)
})
missing[missing["missing_count"] > 0].sort_values("missing_pct", ascending=False)

,missing_count,missing_pct
months_in_default,578825,83.6
next_payment_date_local,470685,67.9
early_repaid_at,463697,66.9
customer_risk_rating,338379,48.8
combined_income,273154,39.4
has_default_within_12_months,37141,5.4
months_on_book,8488,1.2
is_default,7713,1.1
principal_paid_total,7713,1.1
principal_debt,7713,1.1


In [265]:
df[df["is_default"].isna()]    # Comparing is_Default column related to others columns to see if there is any pattern in missing values

,loan_id,country,loan_issued_at,early_repaid_at,issued_amount,loan_status,initial_interest_rate,principal_balance,principal_debt,principal_paid_total,...,is_default,next_payment_date_local,months_in_default,months_on_book,loan_status_risk,repaid_amount_total,initial_loan_duration,combined_income,has_default_within_12_months,customer_risk_rating
333,01EB1FD2-0510-4B7F-9E5E-A4B800BC9FF5,Finland,2015-06-15 23:05:15,2015-06-16 09:17:38,3000.0,Returned,0.1872,NaN,NaN,NaN,...,<NA>,NaN,NaN,NaN,Cancelled,NaN,60,NaN,0,NaN
483,0509BABF-17D9-4EB4-9E67-A4B2017192AA,Spain,2015-06-13 11:40:49,2015-06-15 10:48:34,1000.0,Returned,0.5709,NaN,NaN,NaN,...,<NA>,NaN,NaN,NaN,Cancelled,NaN,60,NaN,0,NaN
574,06E7E6D8-48FB-487C-A054-A38901492C80,Spain,2014-08-20 09:00:00,2014-08-25 13:20:01,1000.0,Repaid,0.3400,NaN,NaN,NaN,...,<NA>,NaN,NaN,0.0,Cancelled,NaN,12,NaN,0,NaN
611,07BEF4BA-CF83-49ED-AE33-A3CA00E01BD9,Spain,2014-10-24 16:55:09,2014-11-21 01:30:14,1800.0,Repaid,0.3100,NaN,NaN,NaN,...,<NA>,NaN,NaN,0.0,Paid Up,NaN,60,NaN,0,NaN
653,08A3A0DA-46FE-45F0-BA3C-3EB2F1CCFF50,Estonia,2015-04-12 17:30:00,2015-04-29 01:30:05,500.0,Repaid,0.2596,NaN,NaN,NaN,...,<NA>,NaN,NaN,0.0,Paid Up,NaN,48,NaN,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
843373,F80FB4C9-74D5-407D-961E-2B24D4DCED9C,Estonia,2015-03-17 02:40:07,42103.062581,500.0,Repaid,0.4290,NaN,NaN,NaN,...,<NA>,NaN,NaN,0.0,Paid Up,NaN,12,NaN,0,NaN
843479,FA5C28B4-6C89-4587-8EB0-A4EC011F37FE,Spain,2015-08-12 10:15:32,42236.062685,2000.0,Repaid,0.3112,NaN,NaN,NaN,...,<NA>,NaN,NaN,NaN,Cancelled,NaN,48,NaN,0,NaN
843514,FAF4AA2C-C74B-4EB9-9B79-A37F013A61D9,Slovakia,2014-08-08 20:20:00,41879.572118,1400.0,Repaid,0.2900,NaN,NaN,NaN,...,<NA>,NaN,NaN,0.0,Paid Up,NaN,60,NaN,0,NaN
843648,FD6D4B81-B3F6-447F-ABAD-A4040152641E,Spain,2014-12-27 21:00:00,42028.062593,800.0,Repaid,0.3500,NaN,NaN,NaN,...,<NA>,NaN,NaN,0.0,Paid Up,NaN,36,NaN,0,NaN


In [267]:
df["default_flag"] = df["is_default"].fillna(0).astype(int)    

**We changed the missing values in is_default to 0, so that we can use it for analysis. We will use this column for analysis instead of is_default column**.

**Because in **Loan_Status** all of the Borrowers who have missing values in the **is_default** column repaid or returned the money back. so the missing values in the is_default column doesn't mean missing. So we changed missing values to 0 which is not Default and they paid back the money**.

**We created a new default_flag column using only '0' and '1'** **( 0 = not default, 1 = default)**

In [268]:
print(df["is_default"].value_counts(dropna=False))
print(df["default_flag"].value_counts())            # counted the values of the new column to see if it is correct and matches the original column

is_default
0       571112
1       113954
<NA>      7713
Name: count, dtype: Int64
default_flag
0    578825
1    113954
Name: count, dtype: int64


In [269]:
df["default_flag"].mean() 

np.float64(0.1644882422821708)

**That Means 13.7% of the loans are defaulted in the default_flag columns**

In [270]:
df[df["combined_income"].isna()]

,loan_id,country,loan_issued_at,early_repaid_at,issued_amount,loan_status,initial_interest_rate,principal_balance,principal_debt,principal_paid_total,...,next_payment_date_local,months_in_default,months_on_book,loan_status_risk,repaid_amount_total,initial_loan_duration,combined_income,has_default_within_12_months,customer_risk_rating,default_flag
0,0001981E-E281-499C-906E-AB6100D6BF73,Estonia,2020-02-14 13:10:19,2021-03-01 08:41:52,637.0,Repaid,0.2866,0.00,0.00,637.00,...,NaN,NaN,12.0,Paid Up,831.70,36,NaN,0,NaN,0
2,000E567F-C487-4E02-805D-A2D101020413,Estonia,2014-02-15 09:40:02,NaN,7800.0,Repaid,0.3000,0.00,0.00,7800.00,...,NaN,NaN,62.0,Paid Up,15291.57,36,NaN,0,NaN,0
6,001706E1-FC54-4D75-8630-AD30007D53EA,Estonia,2021-05-22 09:06:56,NaN,5209.0,Defaulted,0.3438,2188.50,2188.50,3020.50,...,NaN,51.0,13.0,Defaulted,4421.31,60,NaN,0,NaN,1
8,001FE05D-A847-45A5-B842-AA750150A417,Estonia,2019-06-23 20:30:26,NaN,1060.0,Repaid,0.3107,0.00,0.00,355.26,...,NaN,NaN,18.0,Written Off,789.83,60,NaN,0,NaN,0
16,0037ED6E-835F-4962-BBBC-AB4B00698CEE,Finland,2020-01-23 20:21:39,NaN,4150.0,Defaulted,0.2000,3422.04,3422.04,727.96,...,NaN,71.0,10.0,Defaulted,1057.67,60,NaN,1,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
843742,FF9EED08-776C-4DE0-92E0-AE7F01446FBA,Finland,2022-04-22 19:46:41,NaN,1556.0,Repaid,0.1927,0.00,0.00,1177.38,...,NaN,NaN,15.0,Written Off,1415.60,60,NaN,0,NaN,0
843745,FFD3535E-61E0-41C8-A124-ADFB017F6C37,Finland,2021-12-12 01:29:58,NaN,518.0,Repaid,0.1927,0.00,0.00,518.00,...,NaN,NaN,2.0,Paid Up,528.07,3,NaN,0,NaN,0
843749,FFE302F6-3D53-495B-8C21-A86800AF311E,Estonia,2018-01-14 10:49:42,NaN,1485.0,Repaid,0.1957,0.00,0.00,1054.90,...,NaN,NaN,4.0,Paid Up,1054.90,60,NaN,1,NaN,0
843751,FFEFC505-692D-4DD2-9C06-AF2D00B5097C,Finland,2022-10-13 11:24:14,45079.916933,4150.0,Repaid,0.1927,0.00,0.00,4150.00,...,NaN,NaN,7.0,Paid Up,4639.52,60,NaN,0,NaN,0


**Combined_income - Borrower's income, combined from bank statement and registry data (tax)**

**The data dictionary says combined_income is “income combined from bank statement and registry data”. So a value only exists if Bondora actually collected one of those for that borrower**

**The missing rows are older loans (2014, 2019, 2020, 2021, 2022). Most likely Bondora didn’t collect or store income this way back then**

In [271]:
# Share of loans with missing income by year

df.groupby(df["loan_issued_at"].dt.year)["combined_income"].apply(lambda s: s.isna().mean()).round(2)

loan_issued_at
2009    1.00
2010    1.00
2011    1.00
2012    1.00
2013    1.00
2014    1.00
2015    1.00
2016    1.00
2017    1.00
2018    1.00
2019    1.00
2020    1.00
2021    1.00
2022    0.86
2023    0.00
2024    0.00
2025    0.00
Name: combined_income, dtype: float64

**From the year **2009 – 2021** no recording have been made for the combined_income (its 100% Missing). In the year **2022** - 86% of combined_income records are missing and followed by the year **2023 - 2025** - 0%. so that means there is no missing records for the combined_income**.

In [273]:
# Share of loans with missing income by country

df.groupby("country", observed=True)["combined_income"].apply(lambda s: s.isna().mean()).round(2)

country
Denmark        0.00
Estonia        0.67
Finland        0.25
Latvia         0.00
Lithuania      1.00
Netherlands    0.00
Slovakia       1.00
Spain          0.94
Name: combined_income, dtype: float64

**followed by the combined_income as mentioned in the above markdown cell, from the **year 2022** they started to record the combined_income and rating system**

**countries where Bondora lent mainly in the early years show high missing shares. It doesn’t mean some countries hide income**

In [274]:
# Is income missing for the same loans as the rating?

pd.crosstab(df["combined_income"].isna(), df["customer_risk_rating"].isna())

customer_risk_rating,False,True
combined_income,,
False,354399,65226
True,1,273153


### Income vs. rating: missing for the same loans

*(All loans, before the 2022–2025 filter)*

- 354,399 loans have both customer_risk_rating and Comboined_income · 273,153 have neither · 65,226 have income only · 1 has rating only.
- When income is missing, the rating is missing too, because both were not recorded for older loans.
- We cannot fill these values, so we keep them empty and analyse income and rating only for loans from **2022 onwards**.

**When income is missing, the rating is missing too. Both started being recorded around the same time**. 

**The Missing values cannot be filled from the data. We keep them as missing and analyse income and rating only for loans issued from 2022 onwards** 

**Analysing data's from 2022 to 2025**

In [275]:
df = df[(df["loan_issued_at"] >= "2022-01-01") & (df["loan_issued_at"] < "2026-01-01")].copy()

print("First loan:", df["loan_issued_at"].min())
print("Last loan:", df["loan_issued_at"].max())
print("Rows and columns:", df.shape)

First loan: 2022-01-01 00:01:19
Last loan: 2025-12-31 23:26:15.567000
Rows and columns: (474461, 22)


In [283]:

# Recheck missing values in the filtered dataset from 2022 to 2025

missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(1)
})
missing[missing["missing_count"] > 0].sort_values("missing_pct", ascending=False)

,missing_count,missing_pct
months_in_default,354503,86.3
early_repaid_at,278774,67.9
next_payment_date_local,223713,54.4
customer_risk_rating,56467,13.7
has_default_within_12_months,37141,9.0
months_on_book,2487,0.6
combined_income,2,0.0


**Analysing data's from 2023 to 2025**

In [284]:
df = df[(df["loan_issued_at"] >= "2023-01-01") & (df["loan_issued_at"] < "2026-01-01")].copy()

print("First loan:", df["loan_issued_at"].min())
print("Last loan:", df["loan_issued_at"].max())
print("Rows and columns:", df.shape)

First loan: 2023-01-01 00:42:48
Last loan: 2025-12-31 23:26:15.567000
Rows and columns: (410867, 23)


In [285]:

# Recheck missing values in the filtered dataset from 2023 to 2025

missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(1)
})
missing[missing["missing_count"] > 0].sort_values("missing_pct", ascending=False)

,missing_count,missing_pct
months_in_default,354503,86.3
early_repaid_at,278774,67.9
next_payment_date_local,223713,54.4
customer_risk_rating,56467,13.7
has_default_within_12_months,37141,9.0
months_on_book,2487,0.6
combined_income,2,0.0


In [286]:
df["months_in_default_flag"] = df["months_in_default"].fillna(0).astype(int)

In [287]:
print(df["months_in_default"].value_counts(dropna=False))    # Counting the values of the months_in_default column to see how many defaulted and non defaulted
print(df["months_in_default_flag"].value_counts()) 

months_in_default
NaN     354503
2.0       3120
6.0       3034
10.0      2927
3.0       2842
1.0       2773
5.0       2765
0.0       2716
7.0       2625
8.0       2559
9.0       2453
4.0       2363
11.0      2173
15.0      2170
12.0      2047
13.0      2004
19.0      1915
17.0      1909
14.0      1846
16.0      1835
18.0      1508
20.0      1114
22.0      1084
21.0      1066
23.0       831
24.0       596
27.0       420
26.0       372
25.0       350
31.0       326
34.0       284
32.0       269
29.0       268
28.0       239
30.0       225
36.0       224
35.0       220
33.0       219
37.0       194
38.0       184
39.0       154
40.0       100
41.0        36
42.0         5
Name: count, dtype: int64
months_in_default_flag
0     357219
2       3120
6       3034
10      2927
3       2842
1       2773
5       2765
7       2625
8       2559
9       2453
4       2363
11      2173
15      2170
12      2047
13      2004
19      1915
17      1909
14      1846
16      1835
18      1508
20      1114


**Handling `months_in_default`**

`months_in_default` counts how many months a loan has been in default. According to the data dictionary, an empty value means the loan has never defaulted (it was repaid or is still being paid) so these are not real missing values.

We did not fill the empty values with 0, because 0 already means "defaulted less than a month ago" (2,716 loans). Instead we created `months_in_default_flag`: 1 = the loan has defaulted, 0 = it has never defaulted. The original column is kept for analysing how long defaulted loans stay in default.

In [288]:
df["months_in_default_flag"] = df["months_in_default"].notna().astype(int)      # Checking how many defaulted and non defaulted in months_in_default column

df["months_in_default_flag"].value_counts()

months_in_default_flag
0    354503
1     56364
Name: count, dtype: int64

In [289]:
pd.crosstab(df["default_flag"], df["months_in_default_flag"])  

months_in_default_flag,0,1
default_flag,,
0,354503,0
1,0,56364


**Check: `default_flag` vs `months_in_default_flag`**

The two flags agree for every loan: all 354,503 non-defaulted loans have no `months_in_default` value, and all 56,364 defaulted loans have one. This confirms that an empty `months_in_default` means the loan never defaulted. We use `default_flag` for the analysis.

In [292]:
missing_12m = df[df["has_default_within_12_months"].isna()]

print(len(missing_12m))
print(missing_12m["loan_issued_at"].agg(["min", "max"]))     #Counts the min and max loan issued date for the missing_12m dataset to see if they are recent loans
print(missing_12m["loan_status"].value_counts())             #Counts for the loan_status column to see how many loans are in each status
print(missing_12m["default_flag"].value_counts())            #Counts for the default_flag column to see how many loans are defaulted and non-defaulted

37141
min   2025-10-24 00:13:37.173
max   2025-12-31 23:26:15.567
Name: loan_issued_at, dtype: datetime64[us]
loan_status
Active                 25466
Repaid                  9211
Defaulted               2464
InPostFundingReview        0
Returned                   0
Name: count, dtype: int64
default_flag
0    34677
1     2464
Name: count, dtype: int64


**Why `has_default_within_12_months` is empty**

37,141 loans have no value in `has_default_within_12_months`. All of them were issued between **24 October 2025 and 31 December 2025**, so they were less than 12 months old when the dataset was published (26 October 2026). This matches the data dictionary: the value is only filled once a loan has been on the books for 12 months.

**These loans split by status as follows:

| Loan status | Loans | Is the 12-month outcome known? | Treatment |
|---|---|---|---|
| Defaulted | 2,464 | Yes, they defaulted within their first 12 months | Set to **1** |
| Repaid | 9,211 | Yes, a fully repaid loan can no longer default | Set to **0** |
| Active | 25,466 | No, they could still default before month 12 | Left **empty** |

The 2,464 defaulted loans match exactly the loans with `default_flag = 1` in this group, which confirms the statuses are consistent.

After filling, only the 25,466 active loans remain empty. The 12-month default rate is calculated on loans with a known outcome only**.

In [293]:
mask = df["has_default_within_12_months"].isna()

# Already defaulted → defaulted within 12 months
df.loc[mask & (df["default_flag"] == 1), "has_default_within_12_months"] = 1

# Fully repaid → can no longer default
df.loc[mask & (df["loan_status"] == "Repaid"), "has_default_within_12_months"] = 0

# Check: what is still empty should be active loans only
print(df["has_default_within_12_months"].value_counts(dropna=False))
print(df[df["has_default_within_12_months"].isna()]["loan_status"].value_counts())

has_default_within_12_months
0       350442
1        34959
<NA>     25466
Name: count, dtype: Int64
loan_status
Active                 25466
Defaulted                  0
InPostFundingReview        0
Repaid                     0
Returned                   0
Name: count, dtype: int64


In [296]:
df[df["has_default_within_12_months"].isna()]

,loan_id,country,loan_issued_at,early_repaid_at,issued_amount,loan_status,initial_interest_rate,principal_balance,principal_debt,principal_paid_total,...,months_in_default,months_on_book,loan_status_risk,repaid_amount_total,initial_loan_duration,combined_income,has_default_within_12_months,customer_risk_rating,default_flag,months_in_default_flag
36,008F634F-239E-46C1-98FD-B3A600C00F85,Finland,2025-12-01 11:47:13.503,NaN,2000.0000,Active,0.1748,1926.18000,0.00000,73.82000,...,NaN,10.0,Active,439.51000,120,2636.29000,<NA>,A,0,0
58,00EA5C6D-509B-4847-B429-B3A90085E386,Finland,2025-12-04 12:02:15.280,NaN,11800.0000,Active,0.1748,11423.50000,0.00000,376.50000,...,NaN,10.0,Active,2171.49000,120,2079.64000,<NA>,AA,0,0
131,019A25CD-4FCB-4981-991C-6A9EDFA327DC,Denmark,2025-10-27 15:19:49.083,NaN,690.3485,Active,0.1941,616.07641,8.04826,74.27212,...,NaN,12.0,Active,188.18500,60,2678.28418,<NA>,A,0,0
134,019A45EF-710B-4A22-AFDD-C0DF0E3B2D34,Denmark,2025-11-02 21:15:29.543,NaN,897.4531,Active,0.1941,790.43700,0.00000,107.01609,...,NaN,11.0,Active,263.76810,60,3695.71046,<NA>,A,0,0
135,019A46AC-6747-42F5-9339-676804C71985,Denmark,2025-11-03 00:38:52.330,NaN,1974.3968,Active,0.1941,1738.96515,0.00000,235.43164,...,NaN,11.0,Active,580.30698,60,3684.04826,<NA>,B,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
843586,FC52EE35-0160-4CB5-8DA6-B39D00C2939C,Estonia,2025-11-22 11:54:37.150,NaN,318.0000,Active,0.1999,315.42000,0.00000,2.58000,...,NaN,11.0,Active,63.66000,120,1319.02000,<NA>,AA,0,0
843609,FCBEB91D-86E7-4241-8E5D-B3B300E84A8C,Estonia,2025-12-14 14:08:44.227,NaN,212.0000,Active,0.2299,206.95000,0.00000,5.05000,...,NaN,10.0,Active,52.19000,120,1971.11000,<NA>,A,0,0
843668,FDDE309C-0E20-4F8C-A18C-B39B0109CFF5,Finland,2025-11-20 16:52:13.777,NaN,1600.0000,Active,0.1748,1540.94000,0.00000,59.06000,...,NaN,11.0,Active,358.25000,120,2082.33000,<NA>,B,0,0
843671,FDEC458A-9AF4-4660-B71A-B38E00FC167A,Finland,2025-11-07 19:41:18.230,NaN,200.0000,Active,0.1748,156.10000,0.52000,43.90000,...,NaN,11.0,Active,71.60000,12,1745.85000,<NA>,C,0,0


**The borrowers who are active in the loan status and not has_defualt_within_12 months are considered as NA(Missing Values)**

In [297]:
df["has_default_within_12_months"].mean()  

np.float64(0.09070811959491543)

**It means 9.1% of loans defaulted within their first 12 months**.

In [ ]:
df_12m = df[df["has_default_within_12_months"].notna()].copy()             

print(df_12m.shape)
print("12-month default rate:", round(df_12m["has_default_within_12_months"].mean(), 3))

(385401, 23)
12-month default rate: 0.091


In [299]:
df["default_flag"].mean() 

np.float64(0.13718307870916865)

**That Means 13.7% of the loans are defaulted in the default_flag columns**

**12-month default rate**

We created `df_12m`, which keeps only the 385,401 loans whose 12-month outcome is known. The 25,466 active loans younger than 12 months are excluded, because their outcome is not yet known.

- Overall default rate: **13.7%** (56,364 of 410,867 loans)
- 12-month default rate: **9.1%** (of 385,401 loans)

The 12-month rate is lower because some loans only default after their first year. We use the 12-month rate to compare groups (rating, country, income), because every loan is observed for the same amount of time.

In [295]:
df[df["customer_risk_rating"].isna()]

,loan_id,country,loan_issued_at,early_repaid_at,issued_amount,loan_status,initial_interest_rate,principal_balance,principal_debt,principal_paid_total,...,months_in_default,months_on_book,loan_status_risk,repaid_amount_total,initial_loan_duration,combined_income,has_default_within_12_months,customer_risk_rating,default_flag,months_in_default_flag
1,0009E1E6-1880-44DE-9F3F-AFE9011E58C7,Finland,2023-04-19 18:17:39.000,NaN,4146.0,Repaid,0.1927,0.00,0.0,2438.49,...,NaN,15.0,Written Off,3010.50,60,2124.89,0,NaN,0,0
5,0014F40C-0920-4B6A-898B-AFC1001E4CE0,Finland,2023-03-10 01:54:53.000,2025-02-19 11:48:13.287000,4146.0,Repaid,0.1927,0.00,0.0,4146.00,...,NaN,23.0,Paid Up,5676.59,24,3211.36,0,NaN,0,0
23,005BAF23-5D08-415D-A3D7-AFAA00D85E52,Netherlands,2023-02-15 22:16:45.000,NaN,4040.0,Repaid,0.0920,0.00,0.0,1527.44,...,NaN,15.0,Written Off,1807.93,60,2255.42,0,NaN,0,0
42,00AF28C9-E833-47F2-9F78-B00700A33592,Finland,2023-05-19 13:06:06.000,2023-06-09 12:43:12,2073.0,Repaid,0.1927,0.00,0.0,2073.00,...,NaN,0.0,Paid Up,2092.76,18,1506.26,0,NaN,0,0
80,0156DC66-8795-4EB5-A1D0-AFBF00E1F165,Finland,2023-03-08 13:56:59.000,2023-03-09 12:44:17,518.0,Repaid,0.1927,0.00,0.0,518.00,...,NaN,0.0,Cancelled,518.28,6,3067.74,0,NaN,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
843633,FD3E8092-BEA0-48C2-8404-AFBF01437C73,Finland,2023-03-08 19:41:32.000,44993.999167,3627.0,Repaid,0.1927,0.00,0.0,3627.00,...,NaN,NaN,Cancelled,3627.00,60,2554.08,0,NaN,0,0
843685,FE51E2E1-78C4-4C87-AA42-B01B01055839,Finland,2023-06-08 15:52:36.000,NaN,621.0,Active,0.1927,541.66,0.0,79.34,...,NaN,41.0,Active,504.30,119,1589.31,0,NaN,0,0
843706,FEEAA84A-E149-421F-8B8D-B00600FDFD30,Finland,2023-05-18 15:35:36.000,45300.698995,2487.0,Repaid,0.1927,0.00,0.0,2487.00,...,NaN,7.0,Paid Up,2779.95,60,1507.52,0,NaN,0,0
843713,FF29AD4C-2538-4292-8CC2-B046014B31C2,Estonia,2023-07-21 20:15:16.630,NaN,1382.0,Active,0.1899,682.54,0.0,699.46,...,NaN,39.0,Active,1525.62,60,1942.18,0,NaN,0,0


***Missing customer risk rating***

56,467 loans have no risk rating. Unlike the older loans, these are recent (2023), they have income, and most are from Finland with the same interest rate (19.27%). This suggests a group of loans that Bondora issued without a rating.

**How we handle it**
- We **do not fill** the rating. It is Bondora's own risk assessment and cannot be rebuilt from other columns.
- We **do not delete** these loans. They have all other information and are still used in the analysis.
- We put them in a separate group, **"Unrated"**, so we can compare their default rate with the rated grades (AA to HR).

***These loans are not old. They’re from 2023, they have income, and many are from Finland with the same interest rate (19.27%). So the reason here is different from the older loans: Bondora probably issued a group of loans (maybe a specific product or campaign) without giving them a rating***.


***These loans still have a valid amount even though they are missing values because they have interest rate, country, income and outcome, so they’re useful for every other analysis***

In [ ]:
df["rating_group"] = (                             #created an “Unrated” group for the missing values(keep the original column unchanged):
    df["customer_risk_rating"]
    .cat.add_categories("Unrated")
    .fillna("Unrated")
)

df["rating_group"].value_counts()

rating_group
AA         93332
B          72408
C          59723
A          58692
Unrated    56467
D          35343
E          20268
F           8865
G           5750
HR            19
Name: count, dtype: int64

In [301]:
#comparing default rates, including unrated loans

df.groupby("rating_group", observed=True)["default_flag"].agg(["count", "mean"])

,count,mean
rating_group,,
AA,93332,0.054611
A,58692,0.101360
B,72408,0.136670
C,59723,0.171810
D,35343,0.208245
E,20268,0.225429
F,8865,0.255161
G,5750,0.264696
HR,19,0.000000


In [302]:
df[df["repaid_amount_total"].isna()]

,loan_id,country,loan_issued_at,early_repaid_at,issued_amount,loan_status,initial_interest_rate,principal_balance,principal_debt,principal_paid_total,...,months_on_book,loan_status_risk,repaid_amount_total,initial_loan_duration,combined_income,has_default_within_12_months,customer_risk_rating,default_flag,months_in_default_flag,rating_group


In [303]:
df[df["principal_balance"].isna()]

,loan_id,country,loan_issued_at,early_repaid_at,issued_amount,loan_status,initial_interest_rate,principal_balance,principal_debt,principal_paid_total,...,months_on_book,loan_status_risk,repaid_amount_total,initial_loan_duration,combined_income,has_default_within_12_months,customer_risk_rating,default_flag,months_in_default_flag,rating_group


In [155]:
df[df["principal_debt"].isna()]

,loan_id,country,loan_issued_at,early_repaid_at,issued_amount,loan_status,initial_interest_rate,principal_balance,principal_debt,principal_paid_total,...,months_in_default,months_on_book,loan_status_risk,repaid_amount_total,initial_loan_duration,combined_income,has_default_within_12_months,customer_risk_rating,default_flag,months_in_default_flag


***Analysing Months_On_Book***

In [305]:
mob = df[df["months_on_book"].isna()]
print(len(mob))
print(mob["loan_status"].value_counts())
print(mob["loan_status_risk"].value_counts())
print(mob["loan_issued_at"].dt.year.value_counts())



2487
loan_status
Repaid                 2487
Active                    0
Defaulted                 0
InPostFundingReview       0
Returned                  0
Name: count, dtype: int64
loan_status_risk
Cancelled      2487
Active            0
Defaulted         0
Paid Up           0
Written Off       0
Name: count, dtype: int64
loan_issued_at
2024    1131
2025    1014
2023     342
Name: count, dtype: int64


**These 2,487 loans were cancelled.All 2,487 have loan_status = Repaid and loan_status_risk = Cancelled. The borrower took the loan and returned the money soon after, so the loan never really ran. Since there is no loan running, there are no months to count, which is why months_on_book is empty**

In [306]:
print(df["loan_status_risk"].value_counts())
print(pd.crosstab(df["loan_status_risk"], df["months_on_book"].isna()))
print(df.groupby("loan_status_risk", observed=True)["default_flag"].mean().round(3))

loan_status_risk
Active         187124
Paid Up        136409
Defaulted       56364
Cancelled       16784
Written Off     14186
Name: count, dtype: int64
months_on_book     False  True 
loan_status_risk               
Active            187124      0
Cancelled          14297   2487
Defaulted          56364      0
Paid Up           136409      0
Written Off        14186      0
loan_status_risk
Active         0.0
Cancelled      0.0
Defaulted      1.0
Paid Up        0.0
Written Off    0.0
Name: default_flag, dtype: float64


***Cancelled loans are not real loans, so they’re not part of a credit risk analysis***.

***Written-off loans - the borrower paid back only part of the loan and then stopped paying. Bondora gave up trying to collect the rest and wrote it off as a loss.***

In [ ]:
# Remove cancelled loans (never ran, cannot default)
before = len(df)
df = df[df["loan_status_risk"] != "Cancelled"].copy()
df["loan_status_risk"] = df["loan_status_risk"].cat.remove_unused_categories()

print("Removed:", before - len(df))                                     # 16,784
print("Rows now:", len(df))                                             # 394,083
print("Missing months_on_book:", df["months_on_book"].isna().sum())     # 0
print("Default rate:", round(df["default_flag"].mean() * 100, 1), "%")  # 14.3%

Removed: 16784
Rows now: 394083
Missing months_on_book: 0
Default rate: 14.3 %


## Why we remove cancelled loans but keep written-off loans

- **Cancelled loans (16,784) are removed.** The borrower returned the money shortly after payout, so the loan never actually ran and could never default. Keeping them would count them as "not defaulted" and make the default rate look lower than it really is (13.7% instead of 14.3%).   

- **Written-off loans (14,186) are kept.** These borrowers repaid only part of their loan, and the rest was lost. They are real loans with real losses, so removing them would hide risk. We will confirm with our mentor whether they should also be counted as defaults.

In [308]:
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(1)
})
missing[missing["missing_count"] > 0].sort_values("missing_pct", ascending=False)

,missing_count,missing_pct
months_in_default,337719,85.7
early_repaid_at,278774,70.7
next_payment_date_local,206932,52.5
customer_risk_rating,54578,13.8
has_default_within_12_months,25466,6.5
combined_income,2,0.0


**Missing by design: `early_repaid_at` and `next_payment_date_local`**

- `early_repaid_at` is empty for 278,774 loans (67.9%) because these loans were not paid off early. Empty means "no early repayment", not unknown.
- `next_payment_date_local` is empty for 223,713 loans (54.4%) because repaid or defaulted loans have no next payment. This column only describes the day the data was extracted, so we do not use it in the analysis.